# TODO: You must set checkpoint_directory to where you want to save and load the model

In [1]:
checkpoint_base_directory ="/Users/jameswalker/Programming/Checkpoints4/MappoCheckpoint"
checkpoint_number = 0

In [2]:
import ray 

ray.shutdown()
ray.init() 

2025-04-27 16:03:14,035	INFO worker.py:1852 -- Started a local Ray instance.


Python version:,3.9.17
Ray version:,2.44.1


In [ ]:
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.core.rl_module.default_model_config import DefaultModelConfig
from tqdm import tqdm 
from metadrive.envs.marl_envs.roundabout_rllib_delegator_env import RoundaboutRLLibDelegatorEnv 
from ray.rllib.core.rl_module.multi_rl_module import MultiRLModuleSpec
from ray.rllib.core.rl_module.rl_module import RLModuleSpec
from ray.rllib.connectors.env_to_module import FlattenObservations

# Not sure if sgd_minibatch_size is being handled properly, but train_batch_size and worker_num are for sure
# worker_num = 1
# train_batch_size = int(1024)
# sgd_minibatch_size = max(256, int(train_batch_size/10))

config = ( PPOConfig()
    .environment(
        RoundaboutRLLibDelegatorEnv,
        disable_env_checking=True
    )
    .env_runners(
        num_env_runners=1, 
        num_cpus_per_env_runner=1,
        num_gpus_per_env_runner=0,
        # THIS IS NEW; It comes from their Multi-Agent Rock Paper Scissors example
        env_to_module_connector=lambda env: FlattenObservations(multi_agent=True),
    )
    # .env_runners(
    #     num_env_runners=3, 
    #     num_cpus_per_env_runner=1,
    #     num_gpus_per_env_runner=0
    #     )
    .multi_agent(
        policies={"agent0", "agent1"},
        policy_mapping_fn=lambda agent_id, episode, **kw: agent_id,
    )
    .framework('torch')
    .learners(
        num_learners=1,
        num_cpus_per_learner=1,
    )
    # .learners(
    #     num_learners=2,
    #     num_cpus_per_learner=2,
    #     # num_gpus_per_learner=0,
    # )
    .training(
        lr=1e-3,
    )    
    .rl_module(
        model_config=DefaultModelConfig(
                use_lstm=True,
                # Use a simpler FCNet when we also have an LSTM.
                fcnet_hiddens=[32],
                lstm_cell_size=256,
                max_seq_len=15,
                vf_share_layers=True,
            ),
        rl_module_spec=MultiRLModuleSpec(
            rl_module_specs={
                "agent0": RLModuleSpec(),
                "agent1": RLModuleSpec(),
            }
        ),
    )
    # .evaluation(
    #     # TODO evaluation_interval?
    #     # evaluation_interval=1,
    #     evaluation_num_env_runners=1, 
    #     # evaluation_duration=10000,
    #     # WARNING algorithm_config.py:4593 -- You have specified 1 evaluation workers, but your `evaluation_interval` is 0 or None! Therefore, evaluation doesn't occur automatically with each call to `Algorithm.train()`. Instead, you have to call `Algorithm.evaluate()` manually in order to trigger an evaluation run.
    # )
    .debugging(log_level='INFO') # INFO, DEBUG, ERROR, WARN
)



In [4]:
# config.build() is deprecated, use build_algo() instead
algo = config.build_algo()
# print(f"Type of algo: {type(algo)}")


2025-04-27 16:03:23,644	WARNING algorithm_config.py:4704 -- You are running PPO on the new API stack! This is the new default behavior for this algorithm. If you don't want to use the new API stack, set `config.api_stack(enable_rl_module_and_learner=False,enable_env_runner_and_connector_v2=False)`. For a detailed migration guide, see here: https://docs.ray.io/en/master/rllib/new-api-stack-migration-guide.html
/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/rllib/algorithms/algorithm.py:512: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
`UnifiedLogger` will be removed in Ray 2.7.
  return UnifiedLogger(config, logdir, loggers=None)
/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this wa

# TRAIN THE ALGORITHM

In [ ]:
# This trains the algorithm
# Prints metrics every few iterations
# Saves/checkpoints the algorithm to your disk in a new folder eachtime

iteration_num = 1_000
print_every_num_iterations = 10
save_every_num_iterations = 50
for iteration in tqdm(range(0, iteration_num)):
    results = algo.train()
    if iteration % print_every_num_iterations == 0:
        # Print
        if "env_runners" in results:
            mean_return = results["env_runners"].get("episode_return_mean", float("nan"))
            print(f"Iteration={iteration} Mean Return={mean_return}", end="")

        # Save if multiple of save_every_num_iterations
        if iteration % save_every_num_iterations == 0:
            # Save Algorithm checkpoint.
            checkpoint_directory = checkpoint_base_directory + str(checkpoint_number)
            algo.save_to_path(checkpoint_directory)
            checkpoint_number = checkpoint_number + 1
            print(f"Saved to {checkpoint_directory}")


  0%|          | 0/1000 [00:00<?, ?it/s](MultiAgentEnvRunner pid=61102) [INFO] Assets version: 0.4.3
(MultiAgentEnvRunner pid=61102) [INFO] Known Pipes: CocoaGraphicsPipe
(MultiAgentEnvRunner pid=61102) [INFO] Start Scenario Index: 0, Num Scenarios : 1


(MultiAgentEnvRunner pid=61102) Overall reward 0
(MultiAgentEnvRunner pid=61102) Overall reward -6888.531623793722
(MultiAgentEnvRunner pid=61102) Overall reward 15.150450683241125
(MultiAgentEnvRunner pid=61102) Overall reward 18.34168862082114


  0%|          | 1/1000 [00:35<9:48:42, 35.36s/it]

Saved to /Users/jameswalker/Programming/Checkpoints4/MappoCheckpoint0
iter=0 R=-1708.827770064157
(MultiAgentEnvRunner pid=61102) Overall reward 19.72841297297873
(MultiAgentEnvRunner pid=61102) Overall reward -6221.6144832740765
(MultiAgentEnvRunner pid=61102) Overall reward -5598.566226857538
(MultiAgentEnvRunner pid=61102) Overall reward -131.5931002723883


  0%|          | 2/1000 [01:11<9:58:10, 35.96s/it]

...

  1%|▏         | 13/1000 [07:48<10:07:09, 36.91s/it]

iter=12 R=-3201.7117064184404
(MultiAgentEnvRunner pid=61102) Overall reward -852.8417800319697
(MultiAgentEnvRunner pid=61102) Overall reward -13472.971764275248
(MultiAgentEnvRunner pid=61102) Overall reward -6164.156039018214


  1%|▏         | 13/1000 [07:57<10:03:51, 36.71s/it]


(MultiAgentEnvRunner pid=61102) Overall reward -5391.8729038395795


KeyboardInterrupt: 

# Save Algorithm instance
"An Algorithm instance typically holds a neural network for computing actions, called the policy, the RL environment you want to optimize against, a loss function, an optimizer, and some code describing the algorithm’s execution logic, like determining when to collect samples, when to update your model, etc."

We can checkpoint an Algorithm to save its state then load it into a new Algorithm instance at a later date.

In [6]:
# Save Algorithm checkpoint.
algo.save_to_path(checkpoint_directory)

# Display the Algorithm RLModule to check we loaded the same RLModule later on
print(type(algo))
algo.get_module()

<class 'ray.rllib.algorithms.ppo.ppo.PPO'>


# How to debug AssertionError
AssertionError: Can not call this API after engine initialization!

Run the cell below. It should fix this error. This error gets raised if you press 'ESC' and shutdown the Environment but the renderer doesn't properly shutdown.

In [3]:
env.close()